middleware
middleware provides a way to more tightly control what happens inside the agent . middleware is useful for the following:
1. tracking agent behaviour with logging, analytics, debugging
2. transforming prompts, tool selection, and output formatting
3. adding retry , fallvacks and early terminating logics
4. applying rate limits, guardrails and pii detection



In [1]:
import os
from dotenv import load_dotenv

load_dotenv()
os.environ["GROQ_API_KEY"]=os.getenv("GROQ_API_KEY")

Summarization middleware
Summarization Middleware ek intermediate layer hai jo long input/output ko short summary me convert karta hai.
Ye LLM ko bhejne se pehle context ko compress karta hai.
Chat history aur large documents handle karne me use hota hai.
Token limit exceed hone se bachata hai.
Cost reduce karta hai aur response speed improve karta hai.
LangChain me mainly memory, chains aur agents ke sath use hota hai.

In [2]:
from langchain.agents import create_agent
from langchain.agents.middleware import SummarizationMiddleware
from langgraph.checkpoint.memory import InMemorySaver
from langchain_core.messages import HumanMessage, SystemMessage

In [3]:
#message based summarization
agent=create_agent(    
    model="groq:llama-3.3-70b-versatile",
    checkpointer=InMemorySaver(),
    middleware=[
        SummarizationMiddleware(
            model="groq:llama-3.3-70b-versatile",
            trigger=("messages",10),
            keep=("messages",4)
        )
    ]
)

In [4]:
config={"configurable":{"thread_id":"test-1"}}

In [5]:
questions = [
    "What is 25 × 18 + 120 ÷ 6?",
    "If 40% of a number is 80, what is the original number?",
    "What is the square root and cube root of 144?",
    "Solve: 2x + 5 = 17",
    "Solve: x² - 5x + 6 = 0",
    "A train moves at 60 km/h for 3 hours. What distance does it cover?",
    "Find the area of a triangle with base 10 cm and height 6 cm",
    "Using Pythagoras theorem, if a=3 and b=4, find c",
    "A number multiplied by 5 and increased by 20 gives 70. Find the number",
    "Find the perimeter and area of a rectangle with length 12 and breadth 8"
]

for q in questions:
    response=agent.invoke({"messages":[HumanMessage(content=q)]},config)
    print("Question:", q)
    
    print("Answer:", response["messages"][-1].content)
    print("-" * 50)
    print(len(response["messages"]))

Question: What is 25 × 18 + 120 ÷ 6?
Answer: To solve this problem, we need to follow the order of operations (PEMDAS):

1. Multiply 25 and 18: 25 × 18 = 450
2. Divide 120 by 6: 120 ÷ 6 = 20
3. Add 450 and 20: 450 + 20 = 470

The final answer is 470.
--------------------------------------------------
2
Question: If 40% of a number is 80, what is the original number?
Answer: To find the original number, we can set up an equation:

40% of x = 80

We can convert the percentage to a decimal by dividing by 100:
0.4x = 80

Now, we can solve for x by dividing both sides by 0.4:
x = 80 / 0.4
x = 200

The original number is 200.
--------------------------------------------------
4
Question: What is the square root and cube root of 144?
Answer: To find the square root and cube root of 144:

Square root: √144 = 12 (since 12 × 12 = 144)
Cube root: ∛144 = ∛(12 × 12 × 1) ≈ ∛(12²), but more precisely, ∛144 = ∛(2³ × 2³ × 3²) = ∛(2⁶ × 3²) = 2² × 3 = 4 × 3 = ∛144 = ∛(12²) = ∛(2² × 2² × 3²) = ∛(2⁴ × 3²) 

Token size

In [8]:
from langchain.agents import create_agent
from langchain.agents.middleware import SummarizationMiddleware
from langchain_core.tools import tool
from langchain_core.messages import HumanMessage
from langgraph.checkpoint.memory import InMemorySaver


@tool
def search_hotels(city:str)->str:
    """Search hotels - returns long response to use more tokens"""
    return f"""Hotels in {city}
    1. Grand Hotel
    2. radison hotel
    """


agent= create_agent(

    model="groq:llama-3.3-70b-versatile",
    tools=[search_hotels],
    checkpointer=InMemorySaver(),
    middleware=[
        SummarizationMiddleware(
            model="groq:llama-3.3-70b-versatile",
            trigger=("tokens",550),
            keep=("tokens",200)
        )
    ]


)

In [9]:
config={"configurable":{"thread_id":"test-1"}}
cities=["Paris", "London", "Tokyo", "New York", "Dubai", "Singapore"]

for city in cities:
    response=agent.invoke({
        "messages":[HumanMessage(content=f"find hotels in {city}")]
    },
    config=config
    )
    print(response["messages"])
   

[HumanMessage(content="Here is a summary of the conversation to date:\n\n## SESSION INTENT\nThe user's primary goal is to find hotels in Paris.\n\n## SUMMARY\nThe user requested a search for hotels in Paris, and the tool provided a list of hotels, including the Grand Hotel and the Radisson Hotel. The search was repeated, yielding the same results.\n\n## ARTIFACTS\nNone\n\n## NEXT STEPS\nThe next step would be to refine the search criteria or provide more specific information about the type of hotel the user is looking for, such as price range, location, or amenities, to get more tailored results.", additional_kwargs={'lc_source': 'summarization'}, response_metadata={}, id='29fb8d8f-dc98-458f-9fd1-b1b3d339bb9e'), AIMessage(content='', additional_kwargs={'tool_calls': [{'id': 'vwrxzjbsd', 'function': {'arguments': '{"city":"Paris"}', 'name': 'search_hotels'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 15, 'prompt_tokens': 456, 'total_tokens': 471, 'com

KeyboardInterrupt: 

fraction based
"kitna % context bhar gaya hai ya kitna rakhna hai"
Model max context = 100,000 tokens” ka matlab

👉 Simple words me:

Model ek time pe maximum 100,000 tokens tak ka data dekh sakta hai

In [10]:
from langchain.agents import create_agent
from langchain.agents.middleware import SummarizationMiddleware
from langchain_core.tools import tool
from langchain_core.messages import HumanMessage
from langgraph.checkpoint.memory import InMemorySaver


@tool
def search_hotels(city:str)->str:
    """Search hotels - returns long response to use more tokens"""
    return f"""Hotels in {city}
    1. Grand Hotel
    2. radison hotel
    """


agent= create_agent(

    model="groq:llama-3.3-70b-versatile",
    tools=[search_hotels],
    checkpointer=InMemorySaver(),
    middleware=[
        SummarizationMiddleware(
            model="groq:llama-3.3-70b-versatile",
            trigger=("fraction",0.005),
            keep=("fraction",00.002)
        )
    ]


)

config={"configurable":{"thread_id":"test-1"}}
cities=["Paris", "London", "Tokyo", "New York", "Dubai", "Singapore"]

for city in cities:
    response=agent.invoke({
        "messages":[HumanMessage(content=f"find hotels in {city}")]
    },
    config=config
    )
    print(response["messages"])
   

[HumanMessage(content="Here is a summary of the conversation to date:\n\n## SESSION INTENT\nThe user's primary goal is to find hotels in Paris.\n\n## SUMMARY\nThe user requested a list of hotels in Paris, and the search results consistently returned two hotels: Grand Hotel and Radison Hotel. The search was repeated multiple times with the same results.\n\n## ARTIFACTS\nNone\n\n## NEXT STEPS\nThe next step could be to refine the search criteria, such as adding specific amenities or location preferences, to provide more tailored results. Alternatively, the user could select one of the listed hotels (Grand Hotel or Radison Hotel) for further information or booking.", additional_kwargs={'lc_source': 'summarization'}, response_metadata={}, id='a9dabfc5-3cd3-4347-8bfe-0dc3ff14ff27'), AIMessage(content='', additional_kwargs={'tool_calls': [{'id': '10s31832v', 'function': {'arguments': '{"city":"Paris"}', 'name': 'search_hotels'}, 'type': 'function'}]}, response_metadata={'token_usage': {'comp

BadRequestError: Error code: 400 - {'error': {'message': "Failed to call a function. Please adjust your prompt. See 'failed_generation' for more details.", 'type': 'invalid_request_error', 'code': 'tool_use_failed', 'failed_generation': '<function=search_hotels={"city": "London"}</function>'}}

Human in the loop

pause agent execution for human approval, editing or rejection of tools calls before they execute. human in-loop is useful for the following:

high stakes operations requiring human approval(e.g db write)
compliance workflows where human oversight is mandatory
long-running conversations where human feedback guides the agent


In [14]:
from langchain.agents import create_agent
from langchain.agents.middleware import HumanInTheLoopMiddleware
from langgraph.checkpoint.memory import InMemorySaver

def  read_email_tool(email_id:str)->str:
    """Mock function to read an email by its id"""
    return f"Email content for ID:{email_id}"

def send_email_tool(recipient:str, subject:str,body:str)->str:
    """Mock function to send an email"""
    return f"email sent to {recipient} with subject {subject}"

In [15]:
agent=create_agent(
    model="groq:llama-3.3-70b-versatile",
    tools=[send_email_tool,read_email_tool],
    checkpointer=InMemorySaver(),
    middleware=[
        HumanInTheLoopMiddleware(
            interrupt_on={
                "send_email_tool":{
                    "allowed_decisions":["approve","edit","reject"]
                },
                "read_email_tool":False

            }
        )
    ]
)

In [16]:
config={"configurable":{"thread_id":"test_approve"}}
response=agent.invoke(
    {"messages":[HumanMessage(content="Send email to john@gmail.com with subject 'hello'  and body 'how are you'")]},
    config=config
)

In [17]:
response

{'messages': [HumanMessage(content="Send email to john@gmail.com with subject 'hello'  and body 'how are you'", additional_kwargs={}, response_metadata={}, id='90f7c1bc-1aa8-4b4c-821c-1b9898011c2a'),
  AIMessage(content='', additional_kwargs={'tool_calls': [{'id': '8hfzqgd04', 'function': {'arguments': '{"body":"how are you","recipient":"john@gmail.com","subject":"hello"}', 'name': 'send_email_tool'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 32, 'prompt_tokens': 310, 'total_tokens': 342, 'completion_time': 0.054539189, 'completion_tokens_details': None, 'prompt_time': 0.030551287, 'prompt_tokens_details': None, 'queue_time': 0.160215863, 'total_time': 0.085090476}, 'model_name': 'llama-3.3-70b-versatile', 'system_fingerprint': 'fp_4f6d808339', 'service_tier': 'on_demand', 'finish_reason': 'tool_calls', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--019d7343-533d-76a2-b409-9122953cf8a4-0', tool_calls=[{'name': 'send_email_tool', 'args': {'

In [21]:
#step 2 approve:
from langgraph.types import Command
if "__interrupt__" in response:
    print("pause! Approving...")
    response=agent.invoke(
        Command(
            resume={
                "decisions":[
                    {"type":"approve"}
                ]
            }
        ),
        config=config
    )

    print(f"Result: {response}")

pause! Approving...
Result: {'messages': [HumanMessage(content="Send email to john@gmail.com with subject 'hello'  and body 'how are you'", additional_kwargs={}, response_metadata={}, id='90f7c1bc-1aa8-4b4c-821c-1b9898011c2a'), AIMessage(content='', additional_kwargs={'tool_calls': [{'id': '8hfzqgd04', 'function': {'arguments': '{"body":"how are you","recipient":"john@gmail.com","subject":"hello"}', 'name': 'send_email_tool'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 32, 'prompt_tokens': 310, 'total_tokens': 342, 'completion_time': 0.054539189, 'completion_tokens_details': None, 'prompt_time': 0.030551287, 'prompt_tokens_details': None, 'queue_time': 0.160215863, 'total_time': 0.085090476}, 'model_name': 'llama-3.3-70b-versatile', 'system_fingerprint': 'fp_4f6d808339', 'service_tier': 'on_demand', 'finish_reason': 'tool_calls', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--019d7343-533d-76a2-b409-9122953cf8a4-0', tool_calls=[{'name': 'se

In [22]:
config={"configurable":{"thread_id":"test_edit"}}
response=agent.invoke(
    {"messages":[HumanMessage(content="Send email to wrong@gmail.com with subject 'hello'  and body 'how are you'")]},
    config=config
)

In [23]:
response

{'messages': [HumanMessage(content="Send email to wrong@gmail.com with subject 'hello'  and body 'how are you'", additional_kwargs={}, response_metadata={}, id='33a19e00-cc8a-4620-a189-727217ca4c69'),
  AIMessage(content='', additional_kwargs={'tool_calls': [{'id': 'skj175122', 'function': {'arguments': '{"body":"how are you","recipient":"wrong@gmail.com","subject":"hello"}', 'name': 'send_email_tool'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 32, 'prompt_tokens': 310, 'total_tokens': 342, 'completion_time': 0.064351029, 'completion_tokens_details': None, 'prompt_time': 0.049789797, 'prompt_tokens_details': None, 'queue_time': 0.07902641, 'total_time': 0.114140826}, 'model_name': 'llama-3.3-70b-versatile', 'system_fingerprint': 'fp_dae98b5ecb', 'service_tier': 'on_demand', 'finish_reason': 'tool_calls', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--019d734d-a042-7c92-882c-bf4bf0899520-0', tool_calls=[{'name': 'send_email_tool', 'args': {

In [ ]:
if "__interrupt__" in response:
    print("paused Editing....")
    response=agent.invoke(
        Command(
            resume={
                "decisions":[
                    {
                        "type":"edit",
                        "edited_action":{
                            "name":"send_email_tool",
                            "args":{
                                "recipient":"correct@gmail.com",
                                "subject":"correct subject",
                                "body":"edited before sending"
                            }
                        }
                    }
                ]
            }
        ),
        config=config
    )

SyntaxError: invalid syntax. Perhaps you forgot a comma? (2203005648.py, line 4)